# 99.1 Yamamoto's Method: Exercises

## Contents

**Tools:** CADET-Process (Python).

**Python used:** CSV loading, array slicing, helper functions and parameter fitting.

**Practises:** gradient data, Yamamoto fitting and comparison with a chromatographic simulation.

1. Fit parameters from gradient experiments
2. Compare with simulation (optional)

```{tip}
Run code cells from top to bottom with **Shift+Enter**. After entering, pasting or changing code, run that cell again to apply it, then rerun any later cells that use its results. Setup cells may show no output; they create objects for later cells. Fill empty answer cells as you go.
```

```{note}
**Optional material.** Chapter 99 is not part of the main sequence. Work through it after the related main topic or whenever time permits.
```

In [ ]:
%matplotlib inline

## Exercise 1: fit gradient experiments

Estimate characteristic charge and equilibrium constant from the gradient experiments, then compare the fitted relation with the data.

Import the data given in `./experiments/three_proteins` using the following helper function.
Then, visualize the data.

In [ ]:
import numpy as np
from CADETProcess.tools.yamamoto import GradientExperiment
def create_experiment(file_name, gradient_volume):
    """CSV should have format of [time, salt, protein]."""

    data = np.loadtxt(file_name, delimiter=",")

    time = data[:, 0]
    c_salt = data[:, 1]
    c_protein = data[:, 2:]

    return GradientExperiment(time, c_salt, c_protein, gradient_volume)

In [ ]:
experiment_1 = create_experiment("./experiments/three_proteins/18.8mL.csv", 18.8e-6)
experiment_2 = create_experiment("./experiments/three_proteins/37.6mL.csv", 37.6e-6)
experiment_3 = create_experiment("./experiments/three_proteins/56.4mL.csv", 56.4e-6)
experiment_4 = create_experiment("./experiments/three_proteins/75.2mL.csv", 75.2e-6)

experiments = [experiment_1, experiment_2, experiment_3, experiment_4]

In [ ]:
from CADETProcess.tools.yamamoto import plot_experiments

experiment_1.plot()
experiment_2.plot()
experiment_3.plot()
experiment_4.plot()

Use these column parameters:

- length: $0.1~\text{m}$
- diameter: $7.7~\text{mm}$
- bed porosity: $0.36$
- particle radius: $34 \times 10^{-6}~\text{m}$
- particle porosity: $0.85$

In [ ]:
from CADETProcess.processModel import ComponentSystem
component_system = ComponentSystem(["Salt", "A", "B", "C"])

from CADETProcess.processModel import StericMassAction
binding_model = StericMassAction(component_system)
binding_model.adsorption_rate = [1, 1, 1, 1]
binding_model.desorption_rate = [1, 1, 1, 1]
binding_model.capacity = 822.5
binding_model.steric_factor = [0, 0, 0, 0]

binding_model.reference_liquid_phase_conc = 1000
binding_model.reference_solid_phase_conc = 822.5

In [ ]:
from CADETProcess.processModel import LumpedRateModelWithPores
column = LumpedRateModelWithPores(component_system, "column")
column.binding_model = binding_model
column.length = 0.1
column.diameter = 0.0077
column.bed_porosity = 0.36
column.particle_radius = 34e-6
column.particle_porosity = 0.85

column.axial_dispersion = 1.5e-6
column.film_diffusion = [2e-6, 5e-7, 5e-7, 5e-7]

Apply the Yamamoto method and save the return of the method in a variable. Print the characteristic charge and the equilibrium constant for all proteins in the system.

In [ ]:
from CADETProcess.tools.yamamoto import fit_parameters
yamamoto_results = fit_parameters(experiments, column)

print(yamamoto_results.characteristic_charge)
print(yamamoto_results.k_eq)

Visualize the results of the parameter estimation and check if the results are a good estimate of the experiments.

In [ ]:
yamamoto_results.plot()

It can be recognized that at least one experiment is not in a good alignment with the found parameter.
Check the output of the parameter estimation and decide which experiment to neglect in the estimation.

In [ ]:
print(f"logarithm of peak salt concentration for Protein 1: {yamamoto_results.log_c_salt_at_max_M[:, 0]}")
print(f"logarithm of peak salt concentration for Protein 2: {yamamoto_results.log_c_salt_at_max_M[:, 1]}")
print(f"logarithm of peak salt concentration for Protein 3: {yamamoto_results.log_c_salt_at_max_M[:, 2]}")
print(f"equilibrium constant: {yamamoto_results.k_eq}")

Viewing the data, the third entry related to the third experiment does not fit into the ascending sequence for the peak salt concentration. Remove the third experiment from the parameter estimation and redo the estimation. Also, visualize the results.

In [ ]:
experiments_adapted = [experiment_1, experiment_2, experiment_4]
yamamoto_results_adapted = fit_parameters(experiments_adapted, column)

print(yamamoto_results_adapted.characteristic_charge)
print(yamamoto_results_adapted.k_eq)

In [ ]:
yamamoto_results_adapted.plot()

## Exercise 2 (optional): compare with simulation

This optional extension uses the fitted parameters in a column model. Check the experimental timing and concentration units before comparing curves.

Try to build a simple model which reproduces the experimental results.
The model should consist of an `Inlet`, the column model and an `Outlet`.
Use the values specified in the tables above.

Keep in mind the following steps for creating a model:

1. Configure the remaining unit operations
1. Construct the flow sheet
1. Construct the process
1. Simulation and plotting
1. Comparison to experiment

### Configure the remaining unit operations

In [ ]:
from CADETProcess.processModel import Inlet, Outlet

inlet = Inlet(component_system, name="inlet")
inlet.flow_rate = 8.33e-09

outlet = Outlet(component_system, name="outlet")

Update initial conditions:

In [ ]:
column.q[0] = binding_model.capacity
column.c = [experiment_1.c_salt_start, 0, 0, 0]
column.cp = [experiment_1.c_salt_start, 0, 0, 0]
column.q = [binding_model.capacity, 0, 0, 0]

### Construct the flow sheet

In [ ]:
from CADETProcess.processModel import FlowSheet

flow_sheet = FlowSheet(component_system)
flow_sheet.add_unit(inlet)
flow_sheet.add_unit(column)
flow_sheet.add_unit(outlet)

flow_sheet.add_connection(inlet, column)
flow_sheet.add_connection(column, outlet)

### Construct the process

In [ ]:
from CADETProcess.processModel import Process

flow_rate = 8.33e-09
sample_volume = 50e-9
wash_volume = 15e-6
gradient_volume =18.8e-6
strip_volume = 10e-6

wash_start = sample_volume/flow_rate
gradient_start = sample_volume/flow_rate + wash_volume/flow_rate
strip_start = sample_volume/flow_rate + wash_volume/flow_rate + gradient_volume/flow_rate
t_cycle = sample_volume/flow_rate + wash_volume/flow_rate + gradient_volume/flow_rate + strip_volume/flow_rate

slope = ((experiment_1.c_salt_end-experiment_1.c_salt_start)/(gradient_volume/flow_rate))

process = Process(flow_sheet, "LWE_Lysozyme_18_8mL")
process.cycle_time = t_cycle

process.add_event(
    "load",
    "flow_sheet.inlet.c",
    [experiment_1.c_salt_start, 0.2, 0.24, 0.16],
    0,
)
process.add_event(
    "wash",
    "flow_sheet.inlet.c",
    [experiment_1.c_salt_start, 0, 0, 0],
    wash_start,
)
process.add_event(
    "grad_start",
    "flow_sheet.inlet.c",
    [[experiment_1.c_salt_start, slope], [0, 0], [0, 0], [0, 0]],
    gradient_start
)
process.add_event(
    "strip",
    "flow_sheet.inlet.c",
    [experiment_1.c_salt_end, 0, 0, 0],
    strip_start,
)

### Simulation and plotting

In [ ]:
from CADETProcess.simulator import Cadet

simulator = Cadet()
sim_res = simulator.simulate(process)

from CADETProcess.plotting import SecondaryAxis
sec = SecondaryAxis(
    components="Salt",
    ylabel="$c_{Salt}$"
)
_ = sim_res.solution.column.outlet.plot(secondary_axes=sec)

### Comparison to experiment

In [ ]:
from CADETProcess.comparison import Comparator, SSE
from CADETProcess.processModel import ComponentSystem
from CADETProcess.reference import ReferenceIO

comparator = Comparator()
reference_a = ReferenceIO(
    "Protein_0",
    experiment_1.time,
    experiment_1.c_protein[:, 0],
    flow_rate=flow_rate,
    component_system=ComponentSystem(["A"]),
)
comparator.add_difference_metric(
    SSE(reference_a, components="A"),
    "outlet.outlet",
)

reference_b = ReferenceIO(
    "Protein_1",
    experiment_1.time,
    experiment_1.c_protein[:, 1],
    flow_rate=flow_rate,
    component_system=ComponentSystem(["B"]),
)
comparator.add_difference_metric(
    SSE(reference_b, components="B"),
    "outlet.outlet",
)

reference_c = ReferenceIO(
    "Protein_2",
    experiment_1.time,
    experiment_1.c_protein[:, 2],
    flow_rate=flow_rate,
    component_system=ComponentSystem(["C"]),
)
comparator.add_difference_metric(
    SSE(reference_c, components="C"),
    "outlet.outlet",
)
_ = comparator.plot_comparison(sim_res)